# L2: Agent 核心概念

> 理解什么是 Agent，为什么需要 Agent

In [ ]:
# === 环境设置 ===
import sys
import os
from pathlib import Path

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

# 加载 .env
env_file = Path(project_path) / ".env" if project_path else None
if env_file and env_file.exists():
    with open(env_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

# 验证模块导入
try:
    from backend.agents import BaseAgent, ReActAgent, ConversationalAgent
    from backend.llm import LLMFactory
    print("✓ Agent 模块导入成功")
except ImportError as e:
    print(f"✗ 模块导入失败: {e}")

项目路径: c:\Users\zying\Desktop\pro_agent\agent-learning-project
Python 版本: 3.14.4
✓ Agent 模块导入成功


## 2.1 Agent vs Chatbot

### 核心区别

| 维度 | Chatbot | Agent |
|------|---------|-------|
| **能力边界** | 只能对话 | 能执行动作 |
| **信息来源** | 训练数据 + 输入 | 可调用外部工具 |
| **自主性** | 被动响应 | 主动规划 |
| **记忆** | 限于当前会话 | 可持久化、检索 |
| **工作模式** | 问 → 答 | 思考 → 行动 → 观察 → 循环 |

### 类比理解

```
Chatbot = 图书馆管理员
          ↓
      只能回答问题，不能去书架上拿书

Agent = 私人助理
          ↓
      可以回答问题、查资料、发邮件、订票...
```

### Agent 的四大核心能力

```
┌─────────────────────────────────────────────────┐
│                   Agent                        │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐      │
│  │  Memory  │  │  Tools   │  │ Planning │      │
│  │  记忆    │  │  工具    │  │  规划    │      │
│  └──────────┘  └──────────┘  └──────────┘      │
│         ↓             ↓             ↓          │
│    记住历史      调用外部API    分解复杂任务   │
└─────────────────────────────────────────────────┘
```

- **Memory（记忆）**：记住对话历史、用户偏好
- **Tools（工具）**：调用外部 API/函数（搜索、计算器等）
- **Planning（规划）**：将复杂任务分解为步骤
- **Action（行动）**：执行具体操作

## 2.2 ReAct 模式

**ReAct** = **Re**asoning（推理）+ **Act**ing（行动）

这是目前最流行的 Agent 实现模式，源自论文：
> *ReAct: Synergizing Reasoning and Acting in Language Models* (2022)

### Thought-Action-Observation 循环

```
用户: 北京现在几点？
    ↓
┌─────────────────────────────────────────┐
│ Thought: 我需要查询北京时间           │
│           但我不知道当前时间           │
│    → Action: 调用 get_time 工具        │
├─────────────────────────────────────────┤
│ Observation: 工具返回 2024-01-15 14:30  │
│    → Thought: 现在我可以回答了         │
│           → Action: 返回答案给用户      │
└─────────────────────────────────────────┘
```

### ReAct 的三种行为

| 行为 | 英文 | 说明 |
|------|------|------|
| **思考** | Thought | 分析当前情况，决定下一步 |
| **行动** | Action | 调用工具或给出答案 |
| **观察** | Observation | 获取工具执行结果 |

### ReAct 循环的伪代码

```python
while not done:
    # 1. 思考
    thought = llm.generate(context + "What should I do?")
    
    # 2. 判断是否需要行动
    if needs_tool(thought):
        # 3. 执行工具
        result = execute_tool(thought.tool_name, thought.args)
        # 4. 观察
        context += f"Observation: {result}"
    else:
        # 5. 给出最终答案
        return thought.answer
```

## 2.3 Agent 状态机

了解 Agent 在运行过程中的状态变化：

In [3]:
from backend.agents.base import AgentState

print("Agent 的状态枚举:")
for state in AgentState:
    print(f"  - {state.value}: {state.name}")

Agent 的状态枚举:
  - idle: IDLE
  - thinking: THINKING
  - acting: ACTING
  - done: DONE
  - error: ERROR


### 状态转换图

```
    ┌─────────┐
    │  IDLE   │  空闲状态
    └────┬────┘
         │ 收到请求
         ↓
    ┌─────────┐
    │THINKING │  思考中 (调用 LLM)
    └────┬────┘
         │ 需要工具
         ↓
    ┌─────────┐
    │ ACTING  │  执行中 (调用工具)
    └────┬────┘
         │ 工具返回
         ↓
    ┌─────────┐
    │THINKING │  再次思考 (基于结果)
    └────┬────┘
         │ 完成或出错
         ↓
    ┌─────────┐
    │  DONE   │  完成
    └─────────┘
```

## 2.4 动手练习：创建对话 Agent

### 练习 1: 创建一个简单的对话 Agent

In [4]:
import asyncio
import os
from pathlib import Path

from backend.agents.react_agent import ConversationalAgent
from backend.agents.base import AgentConfig
from backend.llm import LLMFactory

# 获取 API key
def get_deepseek_key():
    project_path = Path(os.getcwd())
    for path in [project_path] + list(project_path.parents):
        env_file = path / ".env"
        if env_file.exists():
            with open(env_file, encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line.startswith("DEEPSEEK_API_KEY="):
                        return line.split("=", 1)[1].strip()
    return ""

async def chat_demo():
    # 创建 LLM 客户端
    api_key = get_deepseek_key()
    llm = LLMFactory.create("deepseek", api_key=api_key)
    
    # 创建对话 Agent
    config = AgentConfig(
        name="my_assistant",
        description="我的第一个 AI 助手",
        system_prompt="你是一个友好的 AI 助手，擅长解释技术概念。",
        temperature=0.7,
    )
    
    agent = ConversationalAgent(config=config, llm=llm)
    
    # 模拟多轮对话
    questions = [
        "什么是 Agent？",
        "它和 Chatbot 有什么区别？",
        "能不能给我举个例子？",
    ]
    
    session_id = "demo_session_001"
    
    for i, question in enumerate(questions, 1):
        print(f"\n{'='*50}")
        print(f"用户第 {i} 轮: {question}")
        response = await agent.run(question, session_id=session_id)
        print(f"Agent: {response.content[:150]}...")

# 运行
await chat_demo()


用户第 1 轮: 什么是 Agent？
Agent: 我们来用通俗易懂的方式解释一下 **Agent（智能体）** 这个概念。

简单来说，**Agent 就是一个能“看、想、做”的软件程序**。它不只是一个被动的工具（比如一个普通的计算器），而是一个主动的、有目标的“数字员工”。

为了让你更好理解，我们可以把它拆解成三个核心能力：

### 1. ...

用户第 2 轮: 它和 Chatbot 有什么区别？
Agent: “它”在这里指的是哪个具体的对象呢？如果“它”是指我（这个 AI 助手）的话，那么“它”和“Chatbot”的区别可以从几个方面来理解：

1. **定义范围不同**：
   - **Chatbot**（聊天机器人）是一个广义的术语，指任何能够通过文本或语音与用户进行对话的软件程序。它可以非常简单，...

用户第 3 轮: 能不能给我举个例子？
Agent: 当然可以！不过我需要你稍微具体一点，告诉我你希望我举哪方面的例子呢？比如：

- **技术概念**（例如：什么是API？什么是递归？）
- **生活场景**（例如：如何制定学习计划？如何做一道菜？）
- **逻辑推理**（例如：如果A>B，B>C，那么A和C的关系？）
- **创意示例**（例如：给...


### 练习 2: 观察 Agent 的状态变化

In [5]:
async def state_demo():
    """演示 Agent 状态的变化"""
    api_key = get_deepseek_key()
    llm = LLMFactory.create("deepseek", api_key=api_key)
    
    config = AgentConfig(
        name="state_observer",
        system_prompt="你是一个简洁的助手。"
    )
    
    agent = ConversationalAgent(config=config, llm=llm)
    
    print("初始状态:")
    print(f"  State: {agent.state}")
    print(f"  Config: {agent.get_state()}")
    
    # 执行一次对话
    print("\n执行对话中...")
    await agent.run("你好", session_id="state_demo")
    
    print("\n对话后状态:")
    print(f"  State: {agent.state}")

await state_demo()

初始状态:
  State: AgentState.IDLE
  Config: {'name': 'state_observer', 'state': 'idle', 'description': '', 'enable_tools': True, 'enable_memory': True}

执行对话中...

对话后状态:
  State: AgentState.DONE


### 练习 3: 理解 ReAct 循环

In [6]:
from backend.agents.react_agent import ReActAgent

async def react_demo():
    """演示 ReAct Agent 的思考-行动-观察循环"""
    api_key = get_deepseek_key()
    llm = LLMFactory.create("deepseek", api_key=api_key)
    
    # 创建 ReAct Agent
    config = AgentConfig(
        name="react_demo",
        system_prompt="""你是一个基于ReAct模式的智能助手。

工作流程：
1. Thought: 分析用户需求
2. Action: 决定是使用工具还是直接回答
3. Observation: 观察结果
4. 重复直到完成""",
        max_iterations=5,
        enable_tools=False,  # 暂时禁用工具
    )
    
    agent = ReActAgent(config=config, llm=llm)
    
    # 观察 Agent 内部结构
    print("ReAct Agent 结构:")
    print(f"  名称: {agent.config.name}")
    print(f"  最大迭代次数: {agent.config.max_iterations}")
    print(f"  启用工具: {agent.config.enable_tools}")
    
    # 执行一个简单的任务
    print("\n执行任务...")
    response = await agent.run(
        "请用三句话解释什么是 Python 异步编程",
        session_id="react_demo"
    )
    
    print(f"\n最终回答:\n{response.content}")

await react_demo()

ReAct Agent 结构:
  名称: react_demo
  最大迭代次数: 5
  启用工具: False

执行任务...

最终回答:
Thought: 用户要求用三句话解释Python异步编程，我需要给出简洁准确的定义和核心概念。

Action: 直接回答

Python异步编程是一种通过`async/await`语法实现并发执行的编程范式，它允许程序在等待I/O操作（如网络请求、文件读写）时暂停当前任务并切换到其他任务，从而提高效率。其核心是事件循环（Event Loop），它负责调度协程（Coroutine）并在任务阻塞时自动切换上下文。异步编程特别适合处理大量并发I/O密集型任务，但无法加速CPU密集型计算。


## 2.5 查看 Agent 配置模型

了解 AgentConfig 的字段：

In [7]:
from backend.agents.base import AgentConfig

print("AgentConfig 类的字段:")
for field_name, field_info in AgentConfig.model_fields.items():
    default = field_info.default if field_info.default is not None else "(required)"
    print(f"  - {field_name}: {field_info.description or '无描述'} (默认值: {default})")

AgentConfig 类的字段:
  - name: 无描述 (默认值: agent)
  - description: 无描述 (默认值: )
  - system_prompt: 无描述 (默认值: 你是一个有用的AI助手。)
  - temperature: 无描述 (默认值: 0.7)
  - max_iterations: 无描述 (默认值: 10)
  - enable_tools: 无描述 (默认值: True)
  - enable_memory: 无描述 (默认值: True)


## 2.6 常见面试问题

**Q: Agent 和 Chatbot 的本质区别是什么？**
- A: Agent 能执行外部行动（工具调用），Chatbot 仅生成文本。
  - Chatbot: 被动响应，信息来源于训练数据
  - Agent: 主动规划，可通过工具获取实时信息

**Q: 什么是 ReAct 模式？**
- A: ReAct = Reasoning + Acting，一种 Agent 实现模式。
  - **Reasoning**: LLM 思考当前情况
  - **Acting**: 决定调用工具或回答
  - **核心**: Thought → Action → Observation 循环

**Q: Agent 的 Memory 有什么作用？**
- A: Memory 解决三个问题：
  1. **上下文连续性**: 记住历史对话
  2. **长期知识**: 存储用户偏好、重要信息
  3. **上下文窗口管理**: 智能选择保留哪些内容

**Q: Agent 如何处理工具调用失败？**
- A: 典型策略：
  1. 重试（带参数调整）
  2. 尝试其他工具
  3. 告知用户并请求澄清
  4. 跳过该步骤，继续执行

**Q: ReAct 中的最大迭代次数如何设置？**
- A: 需要权衡：
  - 太少: 复杂任务无法完成
  - 太多: 浪费 token，可能陷入死循环
  - 典型值: 5-10 次
  - 可配合 early stopping（检测到完成就停止）

**Q: Agent 的 Temperature 参数应该如何设置？**
- A: 根据任务类型：
  - 工具调用: 低温度 (0-0.3)，确保参数格式正确
  - 对话生成: 中等温度 (0.7-1.0)，增加多样性
  - 代码生成: 低温度 (0-0.2)，确保语法正确
  - 创意写作: 高温度 (1.0+)，鼓励创新

**Q: 什么是 Agent 的 Planning 能力？**
- A: Planning 是 Agent 将复杂任务分解为可执行步骤的能力。
  - **简单规划**: 一步到位（直接工具调用）
  - **多步规划**: 任务拆解（ReAct 循环）
  - **动态规划**: 根据执行结果调整后续步骤
  - **实现方式**: CoT（思维链）、ToT（思维树）、RAP（推理+规划）

**Q: Agent 和 Function Calling 有什么区别？**
- A: Function Calling 是 LLM 的能力，Agent 是利用这种能力的架构。
  | | Function Calling | Agent |
  |--|------------------|-------|
  | **本质** | LLM API 功能 | 完整系统架构 |
  | **范围** | 单次调用返回工具参数 | 包含循环、记忆、规划 |
  | **关系** | 组件 | 使用 Function Calling 的系统 |